## Transformations

### 1. Create New Spark Session

In [0]:
cosmos_uri = "https://azurecapstonecosmosdb.documents.azure.com:443/"
cosmos_key = "W5A3oz4nx0hkZUx1QlWXcZOKfO00y9ff7UbLeEAuiEeE17GrRBt6GPD8o6kOFAVHggbSACZ8fCxgACDbiVMtQA=="
database_name = "Cosmosdbcapstone"
container_name = "Weather"

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CosmosDBIntegration") \
    .config("spark.cosmos.accountEndpoint", cosmos_uri) \
    .config("spark.cosmos.accountKey", cosmos_key) \
    .config("spark.cosmos.database", database_name) \
    .config("spark.cosmos.container", container_name) \
    .getOrCreate()

In [0]:
spark

### 2. Read weather data from Cosmos DB

In [0]:

weather_df = spark.read \
    .format("cosmos.oltp") \
    .option("spark.cosmos.accountEndpoint", cosmos_uri) \
    .option("spark.cosmos.accountKey", cosmos_key) \
    .option("spark.cosmos.database", database_name) \
    .option("spark.cosmos.container", container_name) \
    .load()

In [0]:
weather_df.show()

+----------+--------+--------------------+------+-----+--------------------+------+--------+-----+
|time_stamp|pressure|            location|  rain| wind|                  id|clouds|humidity| temp|
+----------+--------+--------------------+------+-----+--------------------+------+--------+-----+
|1543452704|  996.95|  Financial District|  NULL| 9.79|b568b1d8-ed1a-4bb...|  0.74|    0.75| 37.2|
|1543213445| 1014.18|       North Station|  NULL| 1.35|8e5f04c9-07ad-425...|     1|    0.92|40.51|
|1544683501| 1028.29|           North End|  NULL| 5.03|3ea68149-23a4-4ca...|  0.29|    0.56|21.25|
|1545029101| 1005.54|            West End|  NULL|11.03|c4087c38-0f0f-4d9...|     1|    0.91|38.71|
|1545029101| 1005.53|    Haymarket Square|  NULL|11.11|20a30671-8e75-420...|     1|     0.9|38.86|
|1544741102| 1034.68|            West End|  NULL| 2.31|cf8f8839-f1ef-423...|  0.83|    0.64|31.18|
|1543281514| 1005.35|            West End|0.2043|11.33|d33146a9-e95c-40a...|  0.99|     0.9|44.01|
|154328151

In [0]:
weather_df.printSchema()

root
 |-- time_stamp: string (nullable = true)
 |-- pressure: string (nullable = true)
 |-- location: string (nullable = true)
 |-- rain: string (nullable = true)
 |-- wind: string (nullable = true)
 |-- id: string (nullable = false)
 |-- clouds: string (nullable = true)
 |-- humidity: string (nullable = true)
 |-- temp: string (nullable = true)



### 3. Create a temporary view

In [0]:
weather_df.createOrReplaceTempView("weather")

### 3. Shape of Weather Data

In [0]:
%sql
select count(*) as count from weather

count
6276


### 4. Calculate the number of null values in each column

In [0]:
from pyspark.sql.functions import col, sum


null_count_df = weather_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in weather_df.columns])


null_count_df.show()


+----------+--------+--------+----+----+---+------+--------+----+
|time_stamp|pressure|location|rain|wind| id|clouds|humidity|temp|
+----------+--------+--------+----+----+---+------+--------+----+
|         0|       0|       0|5382|   0|  0|     0|       0|   0|
+----------+--------+--------+----+----+---+------+--------+----+



### 5. Weather Data Transformation

In [0]:
from pyspark.sql.functions import col, to_timestamp, when, dayofweek, hour, year, month, dayofmonth, to_date,from_unixtime

# Timestamp and Data Type Casting and Missing Data Handling
weather_df = weather_df.withColumn("time_stamp", from_unixtime(col("time_stamp")))
weather_df = weather_df.withColumn("pressure", col("pressure").cast("float"))\
       .withColumn("rain", when(col("rain").isNull(), 0).otherwise(col("rain").cast("float")))\
       .withColumn("wind", col("wind").cast("float"))\
       .withColumn("clouds", col("clouds").cast("float"))\
       .withColumn("humidity", col("humidity").cast("float"))\
       .withColumn("temp", col("temp").cast("float"))
       
weather_df = weather_df.withColumn("time_stamp", to_timestamp(col("time_stamp")))


# Create new time-based features
weather_df = weather_df.withColumn("year", year(col("time_stamp")))\
       .withColumn("month", month(col("time_stamp")))\
       .withColumn("day_of_month", dayofmonth(col("time_stamp")))\
       .withColumn("day_of_week", dayofweek(col("time_stamp")))\
       .withColumn("hour", hour(col("time_stamp"))) \
       .withColumn("date", to_date(weather_df["time_stamp"]))

# Remove duplicates
weather_df = weather_df.dropDuplicates(["time_stamp", "location"])


### 6. Create new column with the help of UDF

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Define the function to categorize the time period
def categorize_time_period(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

# Register the function as a UDF
categorize_time_period_udf = udf(categorize_time_period, StringType())

# Apply the UDF to create a new column 'time_period'
weather_df = weather_df.withColumn('time_period', categorize_time_period_udf(weather_df['hour']))

# Show the DataFrame with the new column
weather_df.show()


+-------------------+--------+--------------------+-----+-----+--------------------+------+--------+-----+----+-----+------------+-----------+----+----------+-----------+
|         time_stamp|pressure|            location| rain| wind|                  id|clouds|humidity| temp|year|month|day_of_month|day_of_week|hour|      date|time_period|
+-------------------+--------+--------------------+-----+-----+--------------------+------+--------+-----+----+-----+------------+-----------+----+----------+-----------+
|2018-11-26 07:38:33|  1014.2|    Theatre District|  0.0| 1.94|f0e543cc-9207-4ce...|  0.95|    0.93|41.17|2018|   11|          26|          2|   7|2018-11-26|    Morning|
|2018-12-13 09:45:01| 1030.71|           North End|  0.0| 3.94|63ad93ee-2c0e-47d...|  0.39|    0.64|20.12|2018|   12|          13|          5|   9|2018-12-13|    Morning|
|2018-11-27 02:15:20| 1003.15|  Financial District|0.162|13.73|70068d97-897f-481...|   1.0|    0.89|44.65|2018|   11|          27|          3|   

In [0]:
weather_df.printSchema()

root
 |-- time_stamp: timestamp (nullable = true)
 |-- pressure: float (nullable = true)
 |-- location: string (nullable = true)
 |-- rain: float (nullable = true)
 |-- wind: float (nullable = true)
 |-- id: string (nullable = false)
 |-- clouds: float (nullable = true)
 |-- humidity: float (nullable = true)
 |-- temp: float (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- time_period: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum

# Calculate the number of nulls in each column
null_count_df = weather_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in weather_df.columns])


null_count_df.show()

+----------+--------+--------+----+----+---+------+--------+----+----+-----+------------+-----------+----+----+-----------+
|time_stamp|pressure|location|rain|wind| id|clouds|humidity|temp|year|month|day_of_month|day_of_week|hour|date|time_period|
+----------+--------+--------+----+----+---+------+--------+----+----+-----+------------+-----------+----+----+-----------+
|         0|       0|       0|   0|   0|  0|     0|       0|   0|   0|    0|           0|          0|   0|   0|          0|
+----------+--------+--------+----+----+---+------+--------+----+----+-----+------------+-----------+----+----+-----------+



### 7. Dropping duplicate values

In [0]:
weather_df = weather_df.dropDuplicates(['time_stamp','location'])

### 8. Reading Cab rides data from Cosmos DB

In [0]:
cosmos_uri = "https://azurecapstonecosmosdb.documents.azure.com:443/"
cosmos_key = "W5A3oz4nx0hkZUx1QlWXcZOKfO00y9ff7UbLeEAuiEeE17GrRBt6GPD8o6kOFAVHggbSACZ8fCxgACDbiVMtQA=="
database_name = "Cosmosdbcapstone"
container_name = "car_rides"

In [0]:
# Read data from Cosmos DB
cab_df = spark.read \
    .format("cosmos.oltp") \
    .option("spark.cosmos.accountEndpoint", cosmos_uri) \
    .option("spark.cosmos.accountKey", cosmos_key) \
    .option("spark.cosmos.database", database_name) \
    .option("spark.cosmos.container", container_name) \
    .load()

In [0]:
cab_df.printSchema()

root
 |-- name: string (nullable = true)
 |-- cab_type: string (nullable = true)
 |-- time_stamp: string (nullable = true)
 |-- source: string (nullable = true)
 |-- price: string (nullable = true)
 |-- id: string (nullable = false)
 |-- product_id: string (nullable = true)
 |-- surge_multiplier: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- distance: string (nullable = true)



### 9. Cache the dataFrame for efficient querying

In [0]:

cab_df.cache()

# Trigger the cache
cab_df.count()


693071

### 10. Data Type Conversion

In [0]:
from pyspark.sql.functions import col, to_timestamp, date_format, to_date, from_unixtime

# Data type conversion
cab_df1 = cab_df.withColumn("time_stamp", from_unixtime(col("time_stamp") / 1000))\
                .withColumn("price", col("price").cast("float")) \
                .withColumn("surge_multiplier", col("surge_multiplier").cast("int")) \
                .withColumn("distance", col("distance").cast("float")) \
                .withColumn("hour", hour(col("time_stamp"))) \
                .withColumn("date", to_date(col("time_stamp"))) \
                .withColumn("price_per_mile", col("price") / col("distance"))

cab_df1 = cab_df1.withColumn("time_stamp", to_timestamp(col("time_stamp")))


In [0]:
cab_df1.printSchema()

root
 |-- name: string (nullable = true)
 |-- cab_type: string (nullable = true)
 |-- time_stamp: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- price: float (nullable = true)
 |-- id: string (nullable = false)
 |-- product_id: string (nullable = true)
 |-- surge_multiplier: integer (nullable = true)
 |-- destination: string (nullable = true)
 |-- distance: float (nullable = true)
 |-- hour: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- price_per_mile: double (nullable = true)



In [0]:
cab_df1.show(truncate=False)

+---------+--------+-------------------+-------------+-----+------------------------------------+------------------------------------+----------------+-------------+--------+----+----------+------------------+
|name     |cab_type|time_stamp         |source       |price|id                                  |product_id                          |surge_multiplier|destination  |distance|hour|date      |price_per_mile    |
+---------+--------+-------------------+-------------+-----+------------------------------------+------------------------------------+----------------+-------------+--------+----+----------+------------------+
|UberXL   |Uber    |2018-11-30 22:13:01|North End    |12.0 |009e9c53-074d-43cf-aef2-0fbc7a47ed3d|6f72dfc5-27f1-42e8-84db-ccc7a75f6969|1               |West End     |1.11    |22  |2018-11-30|10.810810671486589|
|UberPool |Uber    |2018-11-29 19:18:00|North End    |5.5  |e219e545-a006-4936-a6cc-7d00adf0e418|997acbb5-e102-41e1-b155-9df7de0a73f2|1               |West End 

### 11. Calculate the number of null values in each column

In [0]:
from pyspark.sql.functions import col, sum


null_count_df = cab_df1.select([sum(col(c).isNull().cast("int")).alias(c) for c in cab_df.columns])
null_count_df.show()

+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+
|name|cab_type|time_stamp|source|price| id|product_id|surge_multiplier|destination|distance|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+
|   0|       0|         0|     0|55095|  0|         0|               0|          0|       0|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+



In [0]:
cab_df1.select(["price","distance"]).orderBy("price").show()

+-----+--------+
|price|distance|
+-----+--------+
| NULL|    7.46|
| NULL|     0.8|
| NULL|    1.16|
| NULL|    2.48|
| NULL|    1.11|
| NULL|    2.94|
| NULL|    3.39|
| NULL|    2.84|
| NULL|    3.39|
| NULL|    3.45|
| NULL|    1.07|
| NULL|    1.16|
| NULL|     1.3|
| NULL|    2.67|
| NULL|    1.08|
| NULL|     2.8|
| NULL|     2.8|
| NULL|    2.58|
| NULL|    1.03|
| NULL|    1.57|
+-----+--------+
only showing top 20 rows



### 12. Drop Null Values and Duplicate Values

In [0]:
cab_df1 = cab_df1.dropna()


In [0]:
cab_df1 = cab_df1.dropDuplicates()

In [0]:
cab_df1.count()

637976

In [0]:
from pyspark.sql.functions import col, sum


null_count_df = cab_df1.select([sum(col(c).isNull().cast("int")).alias(c) for c in cab_df1.columns])
null_count_df.show()

+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+
|name|cab_type|time_stamp|source|price| id|product_id|surge_multiplier|destination|distance|hour|date|price_per_mile|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+
|   0|       0|         0|     0|    0|  0|         0|               0|          0|       0|   0|   0|             0|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+



In [0]:
weather_df = weather_df.dropDuplicates()
cab_df1 = cab_df1.dropDuplicates()

In [0]:
from pyspark.sql.functions import *

repeating_combinations = weather_df.groupBy("location", "hour", "date").agg(count("*").alias("count"))
repeating_combinations.show(truncate=False)

+-----------------------+----+----------+-----+
|location               |hour|date      |count|
+-----------------------+----+----------+-----+
|Beacon Hill            |4   |2018-11-27|2    |
|Boston University      |19  |2018-11-28|3    |
|West End               |0   |2018-12-18|1    |
|Back Bay               |10  |2018-12-02|1    |
|Fenway                 |4   |2018-11-29|9    |
|North End              |21  |2018-11-26|2    |
|Theatre District       |17  |2018-12-01|1    |
|Boston University      |1   |2018-12-01|1    |
|Financial District     |13  |2018-12-01|1    |
|North End              |22  |2018-11-30|1    |
|Back Bay               |8   |2018-11-28|6    |
|West End               |21  |2018-11-27|2    |
|Theatre District       |12  |2018-11-30|1    |
|North End              |13  |2018-12-03|1    |
|South Station          |19  |2018-11-26|2    |
|North End              |3   |2018-11-27|2    |
|Northeastern University|6   |2018-12-16|1    |
|South Station          |22  |2018-12-03

In [0]:
weather_df_unique = weather_df.dropDuplicates(["location", "hour", "date"])

In [0]:
from pyspark.sql.functions import *

repeating_combinations = weather_df_unique.groupBy("location", "hour", "date").agg(count("*").alias("count"))
repeating_combinations.show(truncate=False)

+-----------------------+----+----------+-----+
|location               |hour|date      |count|
+-----------------------+----+----------+-----+
|Beacon Hill            |4   |2018-11-27|1    |
|Boston University      |19  |2018-11-28|1    |
|West End               |0   |2018-12-18|1    |
|Back Bay               |10  |2018-12-02|1    |
|Fenway                 |4   |2018-11-29|1    |
|North End              |21  |2018-11-26|1    |
|Theatre District       |17  |2018-12-01|1    |
|Boston University      |1   |2018-12-01|1    |
|Financial District     |13  |2018-12-01|1    |
|North End              |22  |2018-11-30|1    |
|Back Bay               |8   |2018-11-28|1    |
|West End               |21  |2018-11-27|1    |
|Theatre District       |12  |2018-11-30|1    |
|North End              |13  |2018-12-03|1    |
|South Station          |19  |2018-11-26|1    |
|North End              |3   |2018-11-27|1    |
|Northeastern University|6   |2018-12-16|1    |
|South Station          |22  |2018-12-03

### 13. Performing broadcast join

In [0]:
from pyspark.sql.functions import broadcast

# Broadcast the weather DataFrame
broadcast_weather_df = broadcast(weather_df_unique)

# Perform broadcast join
joined_df = cab_df1.join(
    broadcast_weather_df,
    (cab_df1.source == weather_df_unique.location) &  (cab_df1.date == weather_df_unique.date) &
    (cab_df1.hour == weather_df_unique.hour),
    "left"
)


In [0]:
joined_df.columns

['name',
 'cab_type',
 'time_stamp',
 'source',
 'price',
 'id',
 'product_id',
 'surge_multiplier',
 'destination',
 'distance',
 'hour',
 'date',
 'price_per_mile',
 'time_stamp',
 'pressure',
 'location',
 'rain',
 'wind',
 'id',
 'clouds',
 'humidity',
 'temp',
 'year',
 'month',
 'day_of_month',
 'day_of_week',
 'hour',
 'date',
 'time_period']

### 14. Drop the duplicate columns

In [0]:

cleaned_joined_df = joined_df.select(
    cab_df1["*"],  # Select all columns from cab_df1 
     # Select specific columns from weather_df
    weather_df["pressure"],
    weather_df["rain"],
    weather_df["wind"],
    weather_df["clouds"],
    weather_df["humidity"],
    weather_df["temp"],
    weather_df["year"],
    weather_df["month"],
    weather_df["day_of_month"],
    weather_df["day_of_week"],
    weather_df["time_period"]
)
cleaned_joined_df.show()

+------------+--------+-------------------+--------------------+-----+--------------------+--------------------+----------------+-----------------+--------+----+----------+------------------+--------+------+-----+------+--------+-----+----+-----+------------+-----------+-----------+
|        name|cab_type|         time_stamp|              source|price|                  id|          product_id|surge_multiplier|      destination|distance|hour|      date|    price_per_mile|pressure|  rain| wind|clouds|humidity| temp|year|month|day_of_month|day_of_week|time_period|
+------------+--------+-------------------+--------------------+-----+--------------------+--------------------+----------------+-----------------+--------+----+----------+------------------+--------+------+-----+------+--------+-----+----+-----+------------+-----------+-----------+
|        Lyft|    Lyft|2018-11-29 11:02:57|   Boston University|  9.0|d357d0fc-ecda-41d...|                lyft|               1| Theatre District| 

In [0]:

null_count_joined_df = cleaned_joined_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in cleaned_joined_df.columns])
null_count_joined_df.show()

+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+--------+----+----+------+--------+----+----+-----+------------+-----------+-----------+
|name|cab_type|time_stamp|source|price| id|product_id|surge_multiplier|destination|distance|hour|date|price_per_mile|pressure|rain|wind|clouds|humidity|temp|year|month|day_of_month|day_of_week|time_period|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+--------+----+----+------+--------+----+----+-----+------------+-----------+-----------+
|   0|       0|         0|     0|    0|  0|         0|               0|          0|       0|   0|   0|             0|    2734|2734|2734|  2734|    2734|2734|2734| 2734|        2734|       2734|       2734|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+--------+----+----+------+--------+----+----+-----+--------

There are null values in the joined dataset because there are cases when there is no weather data available for a 
particular location and time stamp.

In [0]:
result_df = cleaned_joined_df.dropna()

In [0]:

null_count_joined_df = result_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in result_df.columns])
null_count_joined_df.show()

+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+--------+----+----+------+--------+----+----+-----+------------+-----------+-----------+
|name|cab_type|time_stamp|source|price| id|product_id|surge_multiplier|destination|distance|hour|date|price_per_mile|pressure|rain|wind|clouds|humidity|temp|year|month|day_of_month|day_of_week|time_period|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+--------+----+----+------+--------+----+----+-----+------------+-----------+-----------+
|   0|       0|         0|     0|    0|  0|         0|               0|          0|       0|   0|   0|             0|       0|   0|   0|     0|       0|   0|   0|    0|           0|          0|          0|
+----+--------+----------+------+-----+---+----------+----------------+-----------+--------+----+----+--------------+--------+----+----+------+--------+----+----+-----+--------

### 15. Shape of Final Dataset

In [0]:
# Number of rows
result_df.count()

635242

In [0]:
# Number of columns
len(result_df.columns)

24

In [0]:
write_result_df = result_df.repartition(1)
 
volume_path = "/Volumes/azuredatabricks2239/default/cabrides"
 
write_result_df.write.mode("overwrite").parquet(volume_path)